# Thesis Assets Colab Entrypoint

Thin notebook entrypoint for the repo-root `thesis_assets` scaffold. The default path hints follow the same `sec_ccm_unified_runner.py` conventions used by the local wrapper script:

- `Data_LM/results/sec_ccm_unified_runner/lm2011_post_refinitiv`
- `Data_LM/results/sec_ccm_unified_runner/lm2011_extension`
- latest child run under `Data_LM/results/sec_ccm_unified_runner/finbert_item_analysis/`


In [ ]:
from pathlib import Path
import json
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False
    print("Not running in Colab; skipping drive.mount().")


In [ ]:
def _resolve_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate.resolve()
    return Path("/content/drive")


def _first_existing_path(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


# Edit these paths if your repo clone or Drive mount lives somewhere else.
DRIVE_ROOT = _resolve_drive_root()
REPO_ROOT = _first_existing_path(
    DRIVE_ROOT / "NLP_Thesis",
    Path("/content/NLP_Thesis"),
)
RUN_ID = "colab_demo"
BUILD_COMMAND = "build-all"  # build-all | build-chapter | build-asset
CHAPTER = "chapter5"
ASSET_ID = "ch5_fit_horserace_item7_c0"
DATA_PROFILE = "COLAB_DRIVE"
DRIVE_DATA_ROOT = DRIVE_ROOT / "Data_LM"
OUTPUT_ROOT = DRIVE_DATA_ROOT / "results" / "thesis_assets" / RUN_ID

# Optional explicit overrides. Leave as None to use sec_ccm_unified_runner-style defaults.
LM2011_POST_REFINITIV_DIR = None
LM2011_EXTENSION_DIR = None
FINBERT_RUN_DIR = None


In [ ]:
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from thesis_assets import build_all_assets, build_chapter_assets, build_single_asset
from thesis_assets import resolve_usage_run_paths


In [ ]:
resolved_paths = resolve_usage_run_paths(
    repo_root=REPO_ROOT,
    data_profile=DATA_PROFILE,
    drive_data_root=DRIVE_DATA_ROOT,
    lm2011_post_refinitiv_dir=LM2011_POST_REFINITIV_DIR,
    lm2011_extension_dir=LM2011_EXTENSION_DIR,
    finbert_run_dir=FINBERT_RUN_DIR,
)

print(json.dumps({key: (str(value) if value is not None else None) for key, value in resolved_paths.items()}, indent=2))


In [ ]:
build_kwargs = {
    "run_id": RUN_ID,
    "repo_root": REPO_ROOT,
    "output_root": OUTPUT_ROOT,
    "lm2011_post_refinitiv_dir": resolved_paths["lm2011_post_refinitiv_dir"],
    "lm2011_extension_dir": resolved_paths["lm2011_extension_dir"],
    "finbert_run_dir": resolved_paths["finbert_run_dir"],
}

if BUILD_COMMAND == "build-all":
    result = build_all_assets(**build_kwargs)
elif BUILD_COMMAND == "build-chapter":
    result = build_chapter_assets(chapter=CHAPTER, **build_kwargs)
elif BUILD_COMMAND == "build-asset":
    result = build_single_asset(asset_id=ASSET_ID, **build_kwargs)
else:
    raise ValueError(f"Unsupported BUILD_COMMAND: {BUILD_COMMAND!r}")

payload = {
    "manifest_path": str(result.manifest_path),
    "asset_statuses": {
        asset_id: asset_result.status
        for asset_id, asset_result in sorted(result.asset_results.items())
    },
}
print(json.dumps(payload, indent=2))
